<a href="https://colab.research.google.com/github/Claudiopj88/Atividades_ANN/blob/main/Atividade_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  17.3M      0  0:00:04  0:00:04 --:--:-- 17.3M


In [2]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [3]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
from tensorflow.keras.layers import TextVectorization
max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [5]:
!pip install transformers

In [6]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
results = pipe(["This is a great movie", "This is a bad movie"])
for result in results:
  print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9957
label: NEGATIVE, with score: 0.995


In [8]:

from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
# Iterate over the original test_ds which yields (text_string_tensor, label_tensor)
for batch, (texts, targets) in enumerate(test_ds):
  # Process each individual text and its corresponding target label in the batch
  for i in range(len(texts)):
    # Extract the text string and decode it from bytes to utf-8 string
    current_text = texts[i].numpy().decode('utf-8')
    # Pass the string to the pipeline, truncating if necessary
    yp = pipe(current_text, truncation=True, max_length=512) # pipe expects a string or list of strings

    # Get the true label: targets[i] is a scalar tensor. Convert to int, 1 for positive, 0 for negative
    # Assuming `1` corresponds to positive class in `test_ds` (e.g., from text_dataset_from_directory)
    true_label = int(targets[i].numpy() == 1)
    test_labels.append(true_label)

    # Get the predicted label from the pipeline output
    # 'lvwerra/distilbert-imdb' outputs 'POSITIVE' or 'NEGATIVE'
    predicted_label_is_positive = (yp[0]['label'] == 'POSITIVE')
    y_pred.append(predicted_label_is_positive)

  print(f"{batch} {time.time()-start:.2f} s - Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("Final Accuracy", accuracy_score(test_labels, y_pred))


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0 0.90 s - Accuracy: 0.7812
1 1.43 s - Accuracy: 0.8281
2 1.86 s - Accuracy: 0.8646
3 2.35 s - Accuracy: 0.8906
4 2.95 s - Accuracy: 0.9000
5 3.54 s - Accuracy: 0.9010
6 4.54 s - Accuracy: 0.9107
7 5.81 s - Accuracy: 0.9141
8 6.76 s - Accuracy: 0.9201
9 7.34 s - Accuracy: 0.9250
10 7.99 s - Accuracy: 0.9261
11 8.47 s - Accuracy: 0.9271
12 8.97 s - Accuracy: 0.9255
13 9.41 s - Accuracy: 0.9263
14 9.89 s - Accuracy: 0.9271
15 10.33 s - Accuracy: 0.9258
16 10.75 s - Accuracy: 0.9283
17 11.22 s - Accuracy: 0.9306
18 11.65 s - Accuracy: 0.9309
19 12.07 s - Accuracy: 0.9297
20 12.51 s - Accuracy: 0.9301
21 13.01 s - Accuracy: 0.9276
22 13.42 s - Accuracy: 0.9293
23 13.89 s - Accuracy: 0.9310
24 14.33 s - Accuracy: 0.9287
25 14.74 s - Accuracy: 0.9267
26 15.14 s - Accuracy: 0.9271
27 15.57 s - Accuracy: 0.9275
28 15.96 s - Accuracy: 0.9278
29 16.66 s - Accuracy: 0.9281
30 18.11 s - Accuracy: 0.9284
31 18.80 s - Accuracy: 0.9258
32 19.43 s - Accuracy: 0.9280
33 20.04 s - Accuracy: 0.9265
34 20